# Data Cleaning For Hate Detection

In [20]:
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

import pickle

In [21]:
df = pd.read_csv("data/jigsaw.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFirst 5 rows:")
print(df.head())

Dataset shape: (31914, 8)

Columns:
Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')

Missing values:
id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

Duplicate rows: 0

First 5 rows:
                 id                                       comment_text  toxic  \
0  41e65f43c180838e  YOU ARE A FAT, GEEKY PRICK WHO HAS NOTHING TO ...      1   
1  cade9e425d75948e  Agent X2: Basically thanks - with a 'little' m...      0   
2  8ff87747403ed2e6  Why are my posts being deleted? \n\nI have tri...      0   
3  a95604c7a938abb3  "\n\n Controlled Demolitions and Common Sense ...      0   
4  2288910537399728  I do not understand your reply.  //Blaxthos ( ...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             1        1       0       1              0  
1            

## Remove duplicates and handle missing comments.

In [22]:
print("Original shape:", df.shape)

df = df.drop_duplicates()

print("After removing duplicates:", df.shape)

print("\nMissing comments:", df["comment_text"].isnull().sum())

df = df.dropna(subset=["comment_text"])

print("After removing missing comments:", df.shape)

print("\nToxic distribution:")
print(df["toxic"].value_counts())

print("\nCleaned data preview:")
print(df[["comment_text", "toxic"]].head())


Original shape: (31914, 8)
After removing duplicates: (31914, 8)

Missing comments: 0
After removing missing comments: (31914, 8)

Toxic distribution:
toxic
0    28811
1     3103
Name: count, dtype: int64

Cleaned data preview:
                                        comment_text  toxic
0  YOU ARE A FAT, GEEKY PRICK WHO HAS NOTHING TO ...      1
1  Agent X2: Basically thanks - with a 'little' m...      0
2  Why are my posts being deleted? \n\nI have tri...      0
3  "\n\n Controlled Demolitions and Common Sense ...      0
4  I do not understand your reply.  //Blaxthos ( ...      0


## Cleaning Actual Text

In [23]:
import re

def clean_text(text):
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove mentions/usernames
    text = re.sub(r"@\w+", "", text)
    
    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [24]:
# Clean comment text
df["clean_text"] = df["comment_text"].apply(clean_text)

print("\nOriginal vs cleaned text:")
print(df[["comment_text", "clean_text"]].head())


Original vs cleaned text:
                                        comment_text  \
0  YOU ARE A FAT, GEEKY PRICK WHO HAS NOTHING TO ...   
1  Agent X2: Basically thanks - with a 'little' m...   
2  Why are my posts being deleted? \n\nI have tri...   
3  "\n\n Controlled Demolitions and Common Sense ...   
4  I do not understand your reply.  //Blaxthos ( ...   

                                          clean_text  
0  you are a fat geeky prick who has nothing to d...  
1  agent x basically thanks with a little more li...  
2  why are my posts being deleted i have tried to...  
3  controlled demolitions and common sense recomm...  
4        i do not understand your reply blaxthos t c  


## Create Target Lable
Since Jigsaw has multiple toxicity categories, for our first LSTM model we'll use the toxic column as the binary target

In [25]:
df["label"] = df["toxic"].astype(int)

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nLabel percentages:")
print(df["label"].value_counts(normalize=True) * 100)


Label distribution:
label
0    28811
1     3103
Name: count, dtype: int64

Label percentages:
label
0    90.276994
1     9.723006
Name: proportion, dtype: float64


In [26]:
df[["clean_text", "label"]].to_csv(
    "data/jigsaw_cleaned.csv",
    index=False
)

print("\nCleaned dataset saved successfully.")


Cleaned dataset saved successfully.


## Tokenizing and Padding
converting the cleaned text into numerical sequences that the LSTM can understand.

In [27]:
# Load cleaned dataset
df = pd.read_csv("data/jigsaw_cleaned.csv")

# Separate text and labels
texts = df["clean_text"].astype(str)
labels = df["label"]

# Create tokenizer
vocab_size = 20000

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(texts)

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences
max_length = 200

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

y = labels.values

print("Vocabulary size:", len(tokenizer.word_index))
print("X shape:", X.shape)
print("y shape:", y.shape)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Vocabulary size: 78278
X shape: (31914, 200)
y shape: (31914,)
Training samples: 25531
Testing samples: 6383


## Save Tokeninzed and Processed Data 

In [28]:
# Save processed datasets
with open("data/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("data/train_data.pkl", "wb") as f:
    pickle.dump((X_train, y_train), f)

with open("data/test_data.pkl", "wb") as f:
    pickle.dump((X_test, y_test), f)

print("\nProcessed data saved successfully.")


Processed data saved successfully.


# Data Cleaning For Hare_Rewriting

In [1]:
import pandas as pd 
import re

In [2]:
df = pd.read_csv( "data/paradetox.csv", sep="\t" ) 
print("Original dataset shape:", df.shape) 
print("\nColumns:") 
print(df.columns.tolist())

Original dataset shape: (11927, 4)

Columns:
['toxic', 'neutral1', 'neutral2', 'neutral3']


In [3]:
df = df[["toxic", "neutral1", "neutral2", "neutral3"]]

In [5]:
print("\nMissing values:") 
print(df.isnull().sum())


Missing values:
toxic           0
neutral1        0
neutral2     5973
neutral3    10065
dtype: int64


In [7]:
df = df.dropna(subset=["toxic"])

In [8]:
df = df.dropna(subset=["neutral1"]) 
df = df[["toxic", "neutral1"]] 
df = df.rename( columns={ "neutral1": "neutral" } )

In [9]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+","",text) # Remove URLs
    text = re.sub(r"\s+"," ",text).strip() # Remove Whitespaces
    df["toxic"] = df["toxic"].apply(clean_text)
    df["neutral"] = df["neutral"].apply(clean_text)

In [10]:
df = df[ (df["toxic"].str.len() > 0) & (df["neutral"].str.len() > 0) ]

In [11]:
df = df.drop_duplicates( subset=["toxic", "neutral"] )

In [12]:
df = df[ df["toxic"] != df["neutral"] ]

In [13]:
df = df.reset_index(drop=True)

In [14]:
print("\nFinal dataset shape:", df.shape)

print("\nFinal Columns:" , df.columns.tolist())

print("\nMissing values:", df.isnull().sum())

print("\nDuplicate pairs:", df.duplicated().sum())


Final dataset shape: (11830, 2)

Final Columns: ['toxic', 'neutral']

Missing values: toxic      0
neutral    0
dtype: int64

Duplicate pairs: 0


In [15]:
print("\nSample rewriting pairs:")

for i in range(min(10, len(df))):
    print("\nToxic:")
    print(df.loc[i, "toxic"])

    print("\nNeutral:")
    print(df.loc[i, "neutral"])



Sample rewriting pairs:

Toxic:
he had steel balls too !

Neutral:
he was brave too!

Toxic:
dude should have been taken to api , he would be right at home with all the other knuckleheads there

Neutral:
It would have been good if he went to api. He would fit in.

Toxic:
im not gonna sell the fucking picture , i just want to contribute to the fucking article .

Neutral:
I'm not gonna sell the picture, i just want to contribute to the article.

Toxic:
the garbage that is being created by cnn and other news agencies is outrageous .

Neutral:
the news that is being created by cnn and other news agencies is outrageous.

Toxic:
the reason they dont exist is because neither is a pathological liar like trump .

Neutral:
The reason they don't exist is because neither lies like trump

Toxic:
i hope they beat each other silly .

Neutral:
I think they beat each other

Toxic:
no good bastards that we are and we are unrepentant about it .

Neutral:
no good people that we are and we are unrepentant

In [16]:
df.to_csv("data/rewriting_pairs.csv", index=False)

print("\n Cleaned Dataset \n Saved file in data")


 Cleaned Dataset 
 Saved file in data
